 In this notebook we want to load the data from the weekly json

 Have a function that prepares the data per bin of the week 

 merges them and saves them as pd dataframe

If the data bin has at least 10 files create a conc dataframe 

If it is overrepresented create a different analysis

Thresholds are Min bin files = 10

Overrepresentation threshold = 45

Thresholds may vary depending on how many files survive preprocessing (seasonality removal, corruption check etc)

For overrepresented data different analysis will occur where different samples will make different conc data.

The above approach will allow us to make more robust decisions for overrepresented bins since we have so many samples of them !


In [ ]:
!pip install mpld3
!pip install mplcursors

!pip install smart_open


In [ ]:
# PyOMA functions
from old_functions import *
from oma_prec import *
from oma_helpers import *
from oma_mass_ssi_helpers import *

# General Functions
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime 

In [ ]:
statistics_dict = fetch_database_stats()


In [ ]:
# Load json

k = json.load(open('weekly_bin_data.json'))


In [ ]:
sorted_dates = sorted(k.keys(), key=lambda x: datetime.strptime(x, '%Y%m%d'))

sorted_dates

In [ ]:
columns_choise = [f'ch{channel}s{sensor}' for channel in range(1,4) for sensor in range(2,7)]


In [ ]:
# Maybe try to make a corruption check for the following files

#v

In [ ]:
# Extract high rpm bins
high_rpm_keys = list(weekly_choice_bins.keys())[5:]

# Create healthy weekly_healthy_bins
weekly_healthy_bins = {key : {} for key in high_rpm_keys}


In [ ]:
def corruption_counters(weekly_choice_bins, statistics_dict):
    # Extract high rpm bins
    high_rpm_keys = list(weekly_choice_bins.keys())[5:]

    # Create healthy weekly_healthy_bins
    weekly_healthy_bins = {key : {} for key in high_rpm_keys}

    ato_count = 0
    tg_count = 0
    corr_count = 0
    
    for key in high_rpm_keys:
        for filename in weekly_choice_bins[key]:
            # Check able to open 
            if statistics_dict[filename]['able_to_open']:
                #check time gap
                if not statistics_dict[filename]['time_gap']:
                    # check if any corr status
                    if not any(statistics_dict[filename]['corruption_status'].values()):
                        weekly_healthy_bins[key][filename] = weekly_choice_bins[key][filename]
                    else:
                        corr_count += 1
                else:
                    tg_count += 1
            else:
                ato_count += 1
    return ato_count, tg_count, corr_count, weekly_healthy_bins

In [ ]:
def count_leaf_values(d):
    count = 0
    if isinstance(d, dict):
        for key, value in d.items():
            if isinstance(value, dict):
                count += count_leaf_values(value)
            else:
                count += 1
    else:
        count += 1
    return count

count_leaf_values(weekly_healthy_bins)

In [ ]:
count_leaf_values(weekly_choice_bins)

In [ ]:
for key in weekly_healthy_bins.keys():
    print(f'{key}: {len(list(weekly_healthy_bins[key].keys()))}')

In [ ]:
# Do the above calculations for each week !

# Check if there is an issue with the timegap figures and other flags

# Update thresholds !!!

In [ ]:
# weekly corruption analysis

list_hfw = []

for week in sorted_dates:
    # set weekly data
    weekly_choice_bins = k[week]
    # get corruption counters
    ato_count, tg_count, corr_count, weekly_healthy_bins = corruption_counters(weekly_choice_bins, statistics_dict)
    # check files remaining in hrpm
    healthy_files_in_week = count_leaf_values(weekly_healthy_bins)
    
    list_hfw.append(healthy_files_in_week)
    print(f'Week: {week} has | ato: {ato_count} | tg: {tg_count} | corr: {corr_count} | remaining files: {healthy_files_in_week} |')
    

In [ ]:
mean(list_hfw)


In [ ]:
data = list_hfw

# Calculate the quartiles
quartiles = np.percentile(data, [25, 50, 75])

# Calculate the mean
mean = np.mean(data)

# Calculate the standard deviation
std_dev = np.std(data)

# Display the distribution
plt.hist(data, bins='auto', alpha=0.7, color='blue')
plt.title('Histogram of Data')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

print("Quartiles: ", quartiles)
print("Mean: ", mean)
print("Standard Deviation: ", std_dev)

In [ ]:
# Include logic for concatenation 

# If at least X files in bin : concatenate  

# If bin overrepresented try different analysis

# 

In [ ]:
# Cases to explore and built upon :

# weeks : '20240129' / 16a over {DONE, s = 6}
#         '20240212' / HRPM 'uniform'  {DONE, s = 10}
#         '20240408' / 10 bin slightly over {DONE, s = 8}

In [ ]:
week_num = '20240408'

# set weekly data
weekly_choice_bins = k[week_num]
# get corruption counters
ato_count, tg_count, corr_count, weekly_healthy_bins = corruption_counters(weekly_choice_bins, statistics_dict)

In [ ]:
print(ato_count, tg_count, corr_count, count_leaf_values(weekly_healthy_bins))

In [ ]:
for key in weekly_healthy_bins.keys():
    print(f'{key} : {len(list(weekly_healthy_bins[key].keys()))}')

In [ ]:
import random

# choose sample size
sample_size = 8

sampled_bins = {}

for key, sub_dict in weekly_healthy_bins.items():
    if len(sub_dict) >= sample_size:
        sampled_bins[key] = dict(random.sample(sub_dict.items(), sample_size))
    else:
        sampled_bins[key] = {}

for key in sampled_bins.keys():
    print(f'{key} : {len(list(sampled_bins[key].keys()))}')

In [ ]:
import os
import pandas as pd

# Initialize
outer_dataframe_dict = {}

# Output directory
output_dir_base = '../../data/PyOMA'

for key in high_rpm_keys:
    sorted_dict = sampled_bins[key]
    df_list = []

    # Remove seasonality for each sample
    for filename in sorted_dict.keys():
        try:
            df = read_data_expand(filename + '.gz')
        except Exception as e:
            print(f'filename : {filename} of key : {key} could not be expanded! Error: {e}')
            continue

        try:
            df_deseasonal = deseasonal_(df, statistics_dict[filename], [0.1, 10], 50, 3)
        except Exception as e:
            print(f'filename : {filename} of key : {key} could not remove seasonality! Error: {e}')
            continue

        try:
            df_deseasonal = df_deseasonal[columns_choise]
        except Exception as e:
            print(f'filename : {filename} of key : {key} could not be indexed! Error: {e}')
            continue

        if not df_deseasonal.isna().any().any():
            df_list.append(df_deseasonal)
            print(f'filename : {filename} of key : {key} finished successfully!')
        else:
            print(f'filename : {filename} of key : {key} had NANs coerced')
            continue
    print(f'Bin {key} finished successfully and completed!')

    # Concatenate Df, save progress
    if df_list:
        outer_dataframe_dict[key] = pd.concat(df_list)

        # Create output directory if it doesn't exist
        week_output_dir = f"{output_dir_base}/{week_num}_conc_5DOF"
        if not os.path.exists(week_output_dir):
            os.makedirs(week_output_dir)

        # Create file path
        file_path = f"{week_output_dir}/{week_num}_conc_5DOF_{key}.csv"
        
        # Save CSV
        outer_dataframe_dict[key].to_csv(file_path, index=False)


In [ ]:
# Above the final else should change

# Make sure you use the same sample size along the files so no bins are dissimilar

# 

In [ ]:
# load and analyze the data

# Try to apply hard criteria from :
# 'Interpretation of stabilization diagrams using density-based clustering algorithm'

# Namely :
